## Setup: imports and NLTK resources

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for resource in [
    "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"
]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from wordcloud import WordCloud

STOP_WORDS = set(stopwords.words("english"))
print("Setup complete. Number of English stopwords:", len(STOP_WORDS))

## Q1. Text preprocessing and tokenization

In [ ]:
paragraph = (
    "Artificial intelligence is changing the way people learn, work, and solve problems. "
    "It helps computers recognize patterns, understand language, and make useful predictions. "
    "I enjoy learning about AI because it combines creativity with logical thinking. "
    "Machine learning models can analyze large amounts of data and discover hidden relationships. "
    "However, AI should be developed responsibly, with attention to fairness, privacy, and safety. "
    "In the future, I hope AI will help people make better decisions and improve everyday life."
)

# 1. Convert to lowercase and remove punctuation using re
clean_text = re.sub(r"[^a-z\s]", "", paragraph.lower())
print("Cleaned text:\n", clean_text)

# 2. Tokenize into words and sentences
word_tokens = word_tokenize(paragraph)
sentence_tokens = sent_tokenize(paragraph)
print("\nWord tokens:\n", word_tokens)
print("\nSentence tokens:")
for i, sentence in enumerate(sentence_tokens, start=1):
    print(f"{i}. {sentence}")

# 3. Compare split() and NLTK word_tokenize()
split_tokens = paragraph.split()
nltk_tokens = word_tokenize(paragraph)
print("\nPython split():", split_tokens)
print("\nNLTK word_tokenize():", nltk_tokens)
print("\nToken counts — split():", len(split_tokens), "| word_tokenize():", len(nltk_tokens))
print("Difference: split() separates on whitespace and keeps punctuation attached; "
      "word_tokenize() generally separates punctuation into its own tokens.")

# 4. Remove stopwords
clean_word_tokens = re.findall(r"\b[a-z]+\b", clean_text)
tokens_no_stopwords = [w for w in clean_word_tokens if w not in STOP_WORDS]
print("\nTokens without stopwords:\n", tokens_no_stopwords)

# 5. Frequency distribution excluding stopwords
freq_dist = FreqDist(tokens_no_stopwords)
print("\nWord frequencies (excluding stopwords):")
for word, count in freq_dist.most_common():
    print(f"{word}: {count}")

plt.figure(figsize=(10, 4))
freq_dist.plot(15, title="Most Common Words (Stopwords Removed)")
plt.show()

## Q2. Regex extraction, stopword removal, stemming and lemmatization

In [ ]:
# 1. Extract alphabet-only words using re.findall()
alphabet_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
print("Alphabet-only words:\n", alphabet_words)

# 2. Remove stopwords
filtered_words = [w for w in alphabet_words if w not in STOP_WORDS]
print("\nAfter stopword removal:\n", filtered_words)

# 3. Porter stemming
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(w) for w in filtered_words]

# 4. WordNet lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_words]

comparison = pd.DataFrame({
    "Original": filtered_words,
    "Stemmed (Porter)": stemmed_words,
    "Lemmatized (WordNet, noun default)": lemmatized_words
})
display(comparison)

print("Stemming example: 'relationships' may be reduced to a word stem.")
print("Lemmatization example: lemmatization aims to return a dictionary form.")
print("\nConclusion:")
print("- Stemming is usually faster and useful for search or rough text matching.")
print("- Lemmatization is more linguistically meaningful, but can need POS information.")
print("- WordNetLemmatizer defaults to nouns; passing POS tags can improve verb/adjective results.")

## Q3. Bag of Words and TF-IDF for three short texts

In [ ]:
texts = [
    "Artificial intelligence helps doctors analyze medical images and detect disease early.",
    "Renewable energy from solar panels and wind turbines can reduce pollution.",
    "Online learning platforms offer flexible courses, videos, and interactive lessons."
]
text_names = ["AI in healthcare", "Renewable energy", "Online learning"]

# 1. Bag of Words using CountVectorizer
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    index=text_names,
    columns=count_vectorizer.get_feature_names_out()
)
print("Bag of Words matrix:")
display(bow_df)

# 2. TF-IDF scores
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=text_names,
    columns=tfidf_vectorizer.get_feature_names_out()
)
print("TF-IDF matrix (rounded):")
display(tfidf_df.round(3))

# 3. Top 3 TF-IDF keywords per text
print("Top 3 TF-IDF keywords in each text:")
for i, name in enumerate(text_names):
    row = tfidf_matrix[i].toarray().ravel()
    top_indices = row.argsort()[::-1][:3]
    top_words = [(tfidf_vectorizer.get_feature_names_out()[j], round(row[j], 3))
                 for j in top_indices if row[j] > 0]
    print(f"{name}: {top_words}")

print("\nInterpretation: words with high TF-IDF scores are relatively important in one text "
      "compared with the other texts. With only three short texts, these keywords are illustrative.")

## Q4. Jaccard and cosine similarity between two technology texts

### Text A: Artificial Intelligence
Artificial intelligence learns patterns from data. AI systems can classify images, understand language, and make predictions. They are used in healthcare, education, and business. Good AI systems need useful data and careful evaluation.

### Text B: Blockchain
Blockchain stores records in linked blocks across a distributed network. Blockchain systems can support transparent transactions and shared record keeping. They are used in finance, supply chains, and digital identity. Good blockchain systems need secure design and careful governance.

In [ ]:
text_a = (
    "Artificial intelligence learns patterns from data. AI systems can classify images, "
    "understand language, and make predictions. They are used in healthcare, education, "
    "and business. Good AI systems need useful data and careful evaluation."
)
text_b = (
    "Blockchain stores records in linked blocks across a distributed network. Blockchain "
    "systems can support transparent transactions and shared record keeping. They are used "
    "in finance, supply chains, and digital identity. Good blockchain systems need secure "
    "design and careful governance."
)

def preprocess(text):
    words = re.findall(r"\b[a-z]+\b", text.lower())
    return [w for w in words if w not in STOP_WORDS]

tokens_a = preprocess(text_a)
tokens_b = preprocess(text_b)
print("Tokens A:", tokens_a)
print("Tokens B:", tokens_b)

# a. Jaccard similarity using sets
set_a, set_b = set(tokens_a), set(tokens_b)
intersection = set_a.intersection(set_b)
union = set_a.union(set_b)
jaccard = len(intersection) / len(union) if union else 0.0
print("\nShared words:", sorted(intersection))
print("Jaccard similarity:", round(jaccard, 4))

# b. Cosine similarity using TF-IDF
similarity_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_pair = similarity_vectorizer.fit_transform([text_a, text_b])
cosine = cosine_similarity(tfidf_pair[0:1], tfidf_pair[1:2])[0, 0]
print("Cosine similarity:", round(float(cosine), 4))

# c. Interpretation
print("\nAnalysis:")
print("- Jaccard compares the overlap of unique words and ignores word frequency.")
print("- TF-IDF cosine compares weighted word vectors and gives less weight to words "
      "that are common across documents.")
print("- Here both texts share general terms such as 'systems', 'data', and 'careful'; "
      "cosine similarity may reflect their shared context while Jaccard treats every unique "
      "word equally. Neither metric understands meaning or synonyms by itself.")

## Q5. Sentiment analysis and word cloud

In [ ]:
reviews = [
    "This phone is excellent, fast, and easy to use. The battery lasts all day.",
    "I love the camera quality and the bright display. Great value for money!",
    "The product is disappointing and slow. It stopped working after two days.",
    "Terrible customer service and poor build quality. I would not recommend it.",
    "The package arrived today. The box contains a phone and a charging cable."
]

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    subjectivity = TextBlob(text).sentiment.subjectivity
    if polarity > 0.1:
        label = "Positive"
    elif polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"
    return polarity, subjectivity, label

sentiment_rows = []
for review in reviews:
    polarity, subjectivity, label = get_sentiment(review)
    sentiment_rows.append({
        "Review": review,
        "Polarity": round(polarity, 3),
        "Subjectivity": round(subjectivity, 3),
        "Classification": label
    })

sentiment_df = pd.DataFrame(sentiment_rows)
display(sentiment_df)

# Word cloud using all positive reviews
positive_reviews = sentiment_df.loc[
    sentiment_df["Classification"] == "Positive", "Review"
].tolist()

if positive_reviews:
    positive_text = " ".join(positive_reviews)
    wordcloud = WordCloud(
        width=900, height=450, background_color="white",
        stopwords=STOP_WORDS, collocations=False
    ).generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Positive Reviews")
    plt.show()
else:
    print("No positive reviews were found, so a word cloud could not be generated.")

print("Polarity ranges from -1 (negative) to +1 (positive).")
print("Subjectivity ranges from 0 (more objective) to 1 (more subjective).")
print("These are lexicon-based estimates and may not correctly interpret sarcasm or context.")

## Q6. Next-word prediction with a simple LSTM

This example uses a short paragraph of approximately 100 words as training data. It creates sequences of tokens and trains a small LSTM to predict the next word. Because the dataset is tiny, generated text is only a demonstration and may not be grammatically reliable.

In [ ]:
training_text = (
    "Technology changes how people communicate learn and work. "
    "Artificial intelligence helps computers discover patterns in data and make predictions. "
    "Machine learning improves when models learn from useful examples. "
    "Responsible technology should protect privacy support fairness and remain reliable. "
    "Students can build strong skills by reading experimenting writing code and solving problems. "
    "Good software is tested carefully and improved through feedback. "
    "In the future technology may help communities access education healthcare and clean energy. "
    "People should use new tools thoughtfully and continue learning throughout their lives."
)

print("Approximate training word count:", len(re.findall(r"\b\w+\b", training_text)))

In [ ]:
# Tokenize with Keras Tokenizer
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense
    from tensorflow.keras.utils import to_categorical
except ImportError as e:
    raise ImportError(
        "TensorFlow is required for Q6. Install it with: %pip install tensorflow"
    ) from e

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
word_sequences = tokenizer.texts_to_sequences([training_text])[0]
index_word = tokenizer.index_word
vocab_size = len(tokenizer.word_index) + 1

# Create progressive input sequences: first word -> next word, first two -> third, etc.
input_sequences = []
for i in range(1, len(word_sequences)):
    input_sequences.append(word_sequences[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
y_cat = to_categorical(y, num_classes=vocab_size)

print("Vocabulary size:", vocab_size)
print("Number of training sequences:", len(input_sequences))
print("X shape:", X.shape, "| y shape:", y_cat.shape)

In [ ]:
# Build and train a small LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=X.shape[1]),
    LSTM(64),
    Dense(vocab_size, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X, y_cat, epochs=100, verbose=0)
print("Training complete.")
print("Final training accuracy:", round(float(history.history["accuracy"][-1]), 3))

In [ ]:
def generate_text(seed_text, next_words=12):
    result = seed_text.lower().strip()
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([result])[0]
        sequence = pad_sequences([sequence], maxlen=X.shape[1], padding="pre")
        predictions = model.predict(sequence, verbose=0)[0]
        # Sample from the most likely few words for slightly varied output
        top_k = min(5, len(predictions))
        top_ids = np.argsort(predictions)[-top_k:]
        probabilities = predictions[top_ids]
        probabilities = probabilities / probabilities.sum()
        next_id = int(np.random.choice(top_ids, p=probabilities))
        next_word = index_word.get(next_id, "")
        if not next_word:
            break
        result += " " + next_word
    return result

seed_word = "technology"
print("Seed:", seed_word)
print("\nGenerated text:")
print(generate_text(seed_word, next_words=12))
print("\nSecond sample:")
print(generate_text("students", next_words=12))
print("\nThird sample:")
print(generate_text("artificial intelligence", next_words=12))